# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wilsonmarundaa-netizen/updated-flyRank-intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row equals performance of a page in 90 days. One row has columns like impressions_30 and impressions_90 with a aggregate trend_direction.

In [1]:
!wget https://raw.githubusercontent.com/wilsonmarundaa-netizen/updated-flyRank-intern/main/data/raw/content_refresh_anonymized.csv


--2026-08-18 23:38:43--  https://raw.githubusercontent.com/wilsonmarundaa-netizen/updated-flyRank-intern/main/data/raw/content_refresh_anonymized.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6727670 (6.4M) [text/plain]
Saving to: ‘content_refresh_anonymized.csv’

content_refresh_ano 100%[===================>]   6.42M  --.-KB/s    in 0.01s   

2026-08-18 23:38:45 (469 MB/s) - ‘content_refresh_anonymized.csv’ saved [6727670/6727670]



In [2]:
import pandas as pd

df = pd.read_csv('content_refresh_anonymized.csv')

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [4]:
df.columns


Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

In [11]:
features =['content_id','impressions_90d',
       'clicks_90d', 'pageviews_90d','cpc','sessions_90d', 'users_90d','impressions_last_30d', 'engagement_rate','search_volume'] #main outputs that determine if a trend is up or down for a page
label = ['trend_direction'] # main aim of the outputs to determine if performance is up or down
context = ['engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days'] # main inputs that determine whether a page get more or less impressions
excluded = ['content_type', 'main_intent', 'word_count','freshness_tier',
       'word_count_tier','impression_tier',
       'position_tier'] # results of the features and dont really provide new information on the data

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Features. Data is available in full. Columns like cpc and search volume have some missing values but its not too significant and patterns can still be learnt. The data covers impressions in 90d and the relevant columns have no missing values.

In [12]:
df_features = df[features]
df_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   content_id            30000 non-null  object 
 1   impressions_90d       30000 non-null  int64  
 2   clicks_90d            30000 non-null  int64  
 3   pageviews_90d         30000 non-null  int64  
 4   cpc                   27532 non-null  float64
 5   sessions_90d          30000 non-null  int64  
 6   users_90d             30000 non-null  int64  
 7   impressions_last_30d  30000 non-null  int64  
 8   engagement_rate       30000 non-null  float64
 9   search_volume         27532 non-null  float64
dtypes: float64(3), int64(6), object(1)
memory usage: 2.3+ MB


The label dataset is full without any missing values.

In [7]:
df_label = df[label]
df_label.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 1 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   trend_direction  30000 non-null  object
dtypes: object(1)
memory usage: 234.5+ KB


The context dataset is full without any missing values and covers the duration window of the required time duration.

In [8]:
df_context = df[context]
df_context.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   engaged_sessions_90d   30000 non-null  int64
 1   ai_sessions_90d        30000 non-null  int64
 2   scroll_events_90d      30000 non-null  int64
 3   days_with_impressions  30000 non-null  int64
 4   days_with_sessions     30000 non-null  int64
 5   clicks_last_30d        30000 non-null  int64
 6   sessions_last_30d      30000 non-null  int64
 7   impressions_prev_30d   30000 non-null  int64
 8   clicks_prev_30d        30000 non-null  int64
 9   sessions_prev_30d      30000 non-null  int64
 10  content_age_days       30000 non-null  int64
dtypes: int64(11)
memory usage: 2.5 MB


excluded dataset contains a lot of missing values and is not consistent.

In [9]:
df_excluded = df[excluded]
df_excluded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   content_type     30000 non-null  object 
 1   main_intent      27626 non-null  object 
 2   word_count       22301 non-null  float64
 3   freshness_tier   30000 non-null  object 
 4   word_count_tier  22301 non-null  object 
 5   impression_tier  30000 non-null  object 
 6   position_tier    30000 non-null  object 
dtypes: float64(1), object(6)
memory usage: 1.6+ MB


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The data may be incomplete but it can never tell the future. The different performances doesnt necessarily mean that theres an imbalance instead its what  Im tryin to measure but if i ever want to make this data userful i must be able to use it to suggest future outcomes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.